In [9]:
import pandas as pd
import json
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

def get_value(d, keys, default="INCONNU"):
    """Extraction sécurisée dans un dictionnaire imbriqué"""
    for key in keys:
        if isinstance(d, dict) and key in d:
            d = d[key]
        else:
            return default
    return d if d is not None else default

def build_clean_dwh():
    raw_path = "raw_data"
    final_data = []

    # 1. On repart du JSON BRUT pour être sûr d'avoir les bonnes valeurs
    for file in os.listdir(raw_path):
        if file.endswith(".json"):
            with open(os.path.join(raw_path, file), 'r', encoding='utf-8') as f:
                content = json.load(f)
                results = content if isinstance(content, list) else content.get("results", [])
                
                for item in results:
                    formality = item.get("formality", {})
                    content_data = formality.get("content", {})
                    
                    # --- EXTRACTION NOM ---
                    # Si Morale
                    nom_morale = get_value(content_data, ["personneMorale", "identite", "entreprise", "denomination"], None)
                    # Si Physique
                    nom_p = get_value(content_data, ["personnePhysique", "identite", "descriptionPersonne", "nom"], "")
                    prenom_p = get_value(content_data, ["personnePhysique", "identite", "descriptionPersonne", "prenoms"], [""])
                    nom_physique = f"{nom_p} {prenom_p[0] if isinstance(prenom_p, list) else prenom_p}".strip()
                    
                    # --- EXTRACTION GÉO ---
                    # On teste adresse morale puis adresse physique
                    ville = get_value(content_data, ["personneMorale", "adresseEntreprise", "adresse", "commune"], None)
                    if not ville or ville == "INCONNU":
                        ville = get_value(content_data, ["personnePhysique", "adresseEntreprise", "adresse", "commune"], "VILLE INCONNUE")
                    
                    cp = get_value(content_data, ["personneMorale", "adresseEntreprise", "adresse", "codePostal"], None)
                    if not cp or cp == "INCONNU":
                        cp = get_value(content_data, ["personnePhysique", "adresseEntreprise", "adresse", "codePostal"], "00000")

                    # --- EXTRACTION SECTEUR (NAF) ---
                    naf = get_value(content_data, ["personneMorale", "identite", "entreprise", "codeApe"], None)
                    if not naf or naf == "INCONNU":
                        naf = get_value(content_data, ["personnePhysique", "identite", "entreprise", "codeApe"], "0000Z")

                    final_data.append({
                        "siren": item.get("siren"),
                        "denomination": nom_morale if nom_morale else (nom_physique if nom_physique else "INCONNU"),
                        "ville": str(ville).upper(),
                        "code_postal": str(cp),
                        "code_naf": str(naf),
                        "forme_juridique": formality.get("formeJuridique")
                    })

    df = pd.DataFrame(final_data)

    # --- 2. CRÉATION DU MODÈLE EN ÉTOILE ---
    
    # DIM_ENTREPRISE
    dim_entreprise = df[['siren', 'denomination', 'forme_juridique']].drop_duplicates('siren')
    
    # DIM_GEOGRAPHIE
    dim_geo = df[['code_postal', 'ville']].drop_duplicates().reset_index(drop=True)
    dim_geo['id_geographie'] = dim_geo.index + 1
    
    # DIM_SECTEUR
    dim_secteur = df[['code_naf']].drop_duplicates().reset_index(drop=True)
    dim_secteur['id_secteur'] = dim_secteur.index + 1
    
    # FACT_FINANCE (On lie tout)
    fact_table = df.merge(dim_geo, on=['code_postal', 'ville']).merge(dim_secteur, on='code_naf')
    fact_table = fact_table[['siren', 'id_geographie', 'id_secteur']]

    # --- 3. CHARGEMENT FINAL ---
    dim_entreprise.to_sql('dim_entreprise', engine, if_exists='replace', index=False)
    dim_geo.to_sql('dim_geographie', engine, if_exists='replace', index=False)
    dim_secteur.to_sql('dim_secteur', engine, if_exists='replace', index=False)
    fact_table.to_sql('fact_finance', engine, if_exists='replace', index=False)

    print("✅ DWH construit proprement à partir des fichiers JSON !")

build_clean_dwh()

✅ DWH construit proprement à partir des fichiers JSON !


In [11]:
import pandas as pd
import json
import os
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

def get_value(d, keys, default=None):
    for key in keys:
        if isinstance(d, dict) and key in d:
            d = d[key]
        else:
            return default
    return d

def get_region(cp):
    """Mapping simplifié pour enrichir la Dim Geographie"""
    if not cp or len(str(cp)) < 2: return "Inconnue", "Inconnu"
    dep = str(cp)[:2]
    regions = {
        '75': ('Paris', 'Île-de-France'), '13': ('Bouches-du-Rhône', 'PACA'),
        '69': ('Rhône', 'Auvergne-Rhône-Alpes'), '33': ('Gironde', 'Nouvelle-Aquitaine'),
        '44': ('Loire-Atlantique', 'Pays de la Loire'), '59': ('Nord', 'Hauts-de-France')
    }
    return regions.get(dep, (f"Département {dep}", "Province"))

def build_pro_dwh():
    raw_path = "raw_data"
    final_data = []

    for file in os.listdir(raw_path):
        if file.endswith(".json"):
            with open(os.path.join(raw_path, file), 'r', encoding='utf-8') as f:
                content = json.load(f)
                results = content if isinstance(content, list) else content.get("results", [])
                
                for item in results:
                    c = get_value(item, ["formality", "content"], {})
                    
                    # --- EXTRACTION NOM (Correction Finale) ---
                    nom = get_value(c, ["personneMorale", "identite", "entreprise", "denomination"])
                    if not nom:
                        # On fouille dans personnePhysique -> identite -> descriptionPersonne
                        desc = get_value(c, ["personnePhysique", "identite", "descriptionPersonne"], {})
                        n = desc.get("nom") or ""
                        p = desc.get("prenoms", [""])[0] if isinstance(desc.get("prenoms"), list) else ""
                        nom = f"{n} {p}".strip()

                    # --- EXTRACTION GEO & SECTEUR ---
                    cp = get_value(c, ["personneMorale", "adresseEntreprise", "adresse", "codePostal"]) or \
                         get_value(c, ["personnePhysique", "adresseEntreprise", "adresse", "codePostal"], "00000")
                    
                    ville = get_value(c, ["personneMorale", "adresseEntreprise", "adresse", "commune"]) or \
                            get_value(c, ["personnePhysique", "adresseEntreprise", "adresse", "commune"], "INCONNUE")

                    act = (get_value(c, ["personneMorale", "etablissementPrincipal", "activites"]) or 
                           get_value(c, ["personnePhysique", "etablissementPrincipal", "activites"]) or [{}])[0]

                    dep, reg = get_region(cp)

                    final_data.append({
                        "siren": item.get("siren"),
                        "denomination": nom if nom else "ENTREPRISE INDIVIDUELLE",
                        "ville": ville.upper(),
                        "code_postal": cp,
                        "departement": dep,
                        "region": reg,
                        "code_naf": act.get("codeApe", "0000Z"),
                        "libelle_activite": act.get("descriptionDetaillee", "Inconnue"),
                        "date_creation": get_value(c, ["natureCreation", "dateCreation"], "2022-01-01")
                    })

    df = pd.DataFrame(final_data)

    # --- MODELISATION & SIMULATION FINANCIÈRE ---
    dim_entreprise = df[['siren', 'denomination']].drop_duplicates()
    dim_geo = df[['code_postal', 'ville', 'departement', 'region']].drop_duplicates().reset_index(drop=True)
    dim_geo['id_geographie'] = dim_geo.index + 1
    
    dim_secteur = df[['code_naf', 'libelle_activite']].drop_duplicates().reset_index(drop=True)
    dim_secteur['id_secteur'] = dim_secteur.index + 1

    # TABLE DE FAITS avec DATA SIMULATION
    fact = df.merge(dim_geo, on=['code_postal', 'ville']).merge(dim_secteur, on='code_naf')
    np.random.seed(42)
    fact['chiffre_affaires'] = np.random.randint(10000, 500000, size=len(fact))
    fact['resultat_net'] = fact['chiffre_affaires'] * np.random.uniform(0.05, 0.15, size=len(fact))
    fact['effectif'] = np.random.randint(1, 50, size=len(fact))
    
    fact_finance = fact[['siren', 'id_geographie', 'id_secteur', 'date_creation', 'chiffre_affaires', 'resultat_net', 'effectif']]

    # CHARGEMENT
    dim_entreprise.to_sql('dim_entreprise', engine, if_exists='replace', index=False)
    dim_geo.to_sql('dim_geographie', engine, if_exists='replace', index=False)
    dim_secteur.to_sql('dim_secteur', engine, if_exists='replace', index=False)
    fact_finance.to_sql('fact_finance', engine, if_exists='replace', index=False)

    print("✅ DWH Pro-Grade généré avec simulations financières et géo enrichie !")

build_pro_dwh()

✅ DWH Pro-Grade généré avec simulations financières et géo enrichie !


In [13]:
import pandas as pd
import json
import os
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

def get_value(d, keys, default=None):
    for key in keys:
        if isinstance(d, dict) and key in d:
            d = d[key]
        else:
            return default
    return d

def get_geo_info(cp):
    """Retourne le département et la région selon le Code Postal"""
    if not cp or len(str(cp)) < 2:
        return "00", "Inconnu", "Inconnue"
    dep_code = str(cp)[:2]
    # Mapping simplifié pour le projet
    mapping = {
        "75": ("Paris", "Île-de-France"), "13": ("Bouches-du-Rhône", "PACA"),
        "69": ("Rhône", "Auvergne-Rhône-Alpes"), "33": ("Gironde", "Nouvelle-Aquitaine"),
        "44": ("Loire-Atlantique", "Pays de la Loire"), "59": ("Nord", "Hauts-de-France"),
        "31": ("Haute-Garonne", "Occitanie"), "67": ("Bas-Rhin", "Grand Est")
    }
    res = mapping.get(dep_code, (f"Département {dep_code}", "Province"))
    return dep_code, res[0], res[1]

def build_final_dwh():
    raw_path = "raw_data"
    final_data = []

    for file in os.listdir(raw_path):
        if file.endswith(".json"):
            with open(os.path.join(raw_path, file), 'r', encoding='utf-8') as f:
                content = json.load(f)
                results = content if isinstance(content, list) else content.get("results", [])
                
                for item in results:
                    c = get_value(item, ["formality", "content"], {})
                    
                    # --- NOM (Extraction Multi-niveaux) ---
                    nom = get_value(c, ["personneMorale", "identite", "entreprise", "denomination"])
                    if not nom:
                        p_physique = c.get("personnePhysique", {})
                        identite = p_physique.get("identite", {})
                        desc = identite.get("descriptionPersonne", {})
                        n = desc.get("nom") or identite.get("nom") or ""
                        p = desc.get("prenoms") or [""]
                        p_str = p[0] if isinstance(p, list) and len(p) > 0 else ""
                        nom = f"{n} {p_str}".strip()

                    # --- GEO ---
                    cp = get_value(c, ["personneMorale", "adresseEntreprise", "adresse", "codePostal"]) or \
                         get_value(c, ["personnePhysique", "adresseEntreprise", "adresse", "codePostal"], "00000")
                    ville = get_value(c, ["personneMorale", "adresseEntreprise", "adresse", "commune"]) or \
                            get_value(c, ["personnePhysique", "adresseEntreprise", "adresse", "commune"], "INCONNUE")
                    
                    dep_code, dep_nom, reg_nom = get_geo_info(cp)

                    # --- SECTEUR ---
                    act = (get_value(c, ["personneMorale", "etablissementPrincipal", "activites"]) or 
                           get_value(c, ["personnePhysique", "etablissementPrincipal", "activites"]) or [{}])[0]

                    final_data.append({
                        "siren": item.get("siren"),
                        "denomination": nom if nom else "ENTREPRISE INDIVIDUELLE",
                        "forme_juridique": get_value(item, ["formality", "formeJuridique"], "1000"),
                        "ville": ville.upper(),
                        "code_postal": cp,
                        "departement": dep_nom,
                        "region": reg_nom,
                        "code_naf": act.get("codeApe", "0000Z"),
                        "libelle_activite": act.get("descriptionDetaillee", "Non renseigné"),
                        "date_creation": get_value(c, ["natureCreation", "dateCreation"], "2023-01-01")
                    })

    df = pd.DataFrame(final_data)

    # --- MODELISATION ---
    # DIM_ENTREPRISE (Réintégration Forme Juridique)
    dim_entreprise = df[['siren', 'denomination', 'forme_juridique']].drop_duplicates()

    # DIM_GEOGRAPHIE (Avec Département et Région)
    dim_geo = df[['code_postal', 'ville', 'departement', 'region']].drop_duplicates().reset_index(drop=True)
    dim_geo['id_geographie'] = dim_geo.index + 1
    
    # DIM_SECTEUR
    dim_secteur = df[['code_naf', 'libelle_activite']].drop_duplicates().reset_index(drop=True)
    dim_secteur['id_secteur'] = dim_secteur.index + 1

    # FACT_FINANCE (Simulation complète pour BI)
    fact = df.merge(dim_geo, on=['code_postal', 'ville']).merge(dim_secteur, on='code_naf')
    np.random.seed(42)
    fact['chiffre_affaires'] = np.random.randint(20000, 800000, size=len(fact))
    fact['resultat_net'] = fact['chiffre_affaires'] * np.random.uniform(0.05, 0.12)
    fact['effectif'] = np.random.randint(1, 30, size=len(fact))
    
    fact_finance = fact[['siren', 'id_geographie', 'id_secteur', 'date_creation', 'chiffre_affaires', 'resultat_net', 'effectif']]

    # CHARGEMENT
    dim_entreprise.to_sql('dim_entreprise', engine, if_exists='replace', index=False)
    dim_geo.to_sql('dim_geographie', engine, if_exists='replace', index=False)
    dim_secteur.to_sql('dim_secteur', engine, if_exists='replace', index=False)
    fact_finance.to_sql('fact_finance', engine, if_exists='replace', index=False)

    print("🚀 DWH Finalisé : Entreprises, Géo, Secteurs et Faits financiers chargés !")

build_final_dwh()

🚀 DWH Finalisé : Entreprises, Géo, Secteurs et Faits financiers chargés !


In [16]:
import pandas as pd
import json
import os
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

def get_value(d, keys, default=None):
    for key in keys:
        if isinstance(d, dict) and key in d:
            d = d[key]
        else:
            return default
    return d

def get_geo_info(cp):
    if not cp or len(str(cp)) < 2:
        return "00", "Inconnu", "Inconnue"
    dep_code = str(cp)[:2]
    mapping = {
        "75": ("Paris", "Île-de-France"), "13": ("Bouches-du-Rhône", "PACA"),
        "69": ("Rhône", "Auvergne-Rhône-Alpes"), "33": ("Gironde", "Nouvelle-Aquitaine"),
        "44": ("Loire-Atlantique", "Pays de la Loire"), "59": ("Nord", "Hauts-de-France"),
        "31": ("Haute-Garonne", "Occitanie"), "67": ("Bas-Rhin", "Grand Est")
    }
    res = mapping.get(dep_code, (f"Département {dep_code}", "Province"))
    return dep_code, res[0], res[1]

def clean_libelle(code_naf, current_libelle):
    """Remplace les libellés génériques par des noms de secteurs propres"""
    mapping_naf = {
        "6243": "Conseil en systèmes et logiciels informatiques",
        "6411": "Activités de banque centrale",
        "702C": "Conseil pour les affaires et la gestion",
        "0180": "Services auxiliaires aux cultures",
        "6704": "Activités de gestion de fonds",
        "3840": "Collecte et traitement des eaux usées",
        "6506": "Intermédiation monétaire et crédit",
        "5904": "Production de films et programmes TV",
        "6447": "Activités des sociétés de holding",
        "7701": "Location de voitures et véhicules légers",
        "3850": "Récupération de déchets triés",
        "0000Z": "Secteur Non Défini"
    }
    # Si le libellé contient le message d'erreur d'INPI ou est générique
    if "reconstitution" in str(current_libelle).lower() or str(current_libelle) in ["Non renseigné", "Inconnue"]:
        return mapping_naf.get(str(code_naf), "Autres activités de services")
    return current_libelle

def build_final_dwh_v2():
    raw_path = "raw_data"
    final_data = []

    print("Extraction et nettoyage des données...")
    for file in os.listdir(raw_path):
        if file.endswith(".json"):
            with open(os.path.join(raw_path, file), 'r', encoding='utf-8') as f:
                content = json.load(f)
                results = content if isinstance(content, list) else content.get("results", [])
                
                for item in results:
                    c = get_value(item, ["formality", "content"], {})
                    
                    # --- NOM ---
                    nom = get_value(c, ["personneMorale", "identite", "entreprise", "denomination"])
                    if not nom:
                        desc = get_value(c, ["personnePhysique", "identite", "descriptionPersonne"], {})
                        n = desc.get("nom") or ""
                        p = desc.get("prenoms", [""])[0] if isinstance(desc.get("prenoms"), list) else ""
                        nom = f"{n} {p}".strip()

                    # --- GEO ---
                    cp = get_value(c, ["personneMorale", "adresseEntreprise", "adresse", "codePostal"]) or \
                         get_value(c, ["personnePhysique", "adresseEntreprise", "adresse", "codePostal"], "00000")
                    ville = get_value(c, ["personneMorale", "adresseEntreprise", "adresse", "commune"]) or \
                            get_value(c, ["personnePhysique", "adresseEntreprise", "adresse", "commune"], "INCONNUE")
                    
                    _, dep_nom, reg_nom = get_geo_info(cp)

                    # --- SECTEUR ---
                    act = (get_value(c, ["personneMorale", "etablissementPrincipal", "activites"]) or 
                           get_value(c, ["personnePhysique", "etablissementPrincipal", "activites"]) or [{}])[0]
                    
                    raw_naf = act.get("codeApe", "0000Z")
                    raw_libelle = act.get("descriptionDetaillee", "Inconnue")

                    final_data.append({
                        "siren": item.get("siren"),
                        "denomination": nom if nom else "ENTREPRISE INDIVIDUELLE",
                        "forme_juridique": get_value(item, ["formality", "formeJuridique"], "1000"),
                        "ville": ville.upper(),
                        "code_postal": cp,
                        "departement": dep_nom,
                        "region": reg_nom,
                        "code_naf": raw_naf,
                        "libelle_activite": clean_libelle(raw_naf, raw_libelle),
                        "date_creation": get_value(c, ["natureCreation", "dateCreation"], "2023-01-01")
                    })

    df = pd.DataFrame(final_data)

    # --- MODELISATION ---
    dim_entreprise = df[['siren', 'denomination', 'forme_juridique']].drop_duplicates('siren')
    
    dim_geo = df[['code_postal', 'ville', 'departement', 'region']].drop_duplicates().reset_index(drop=True)
    dim_geo['id_geographie'] = dim_geo.index + 1
    
    dim_secteur = df[['code_naf', 'libelle_activite']].drop_duplicates('code_naf').reset_index(drop=True)
    dim_secteur['id_secteur'] = dim_secteur.index + 1

    # TABLE DE FAITS
    fact = df.merge(dim_geo, on=['code_postal', 'ville']).merge(dim_secteur, on='code_naf')
    np.random.seed(42)
    fact['chiffre_affaires'] = np.random.randint(20000, 900000, size=len(fact))
    fact['resultat_net'] = (fact['chiffre_affaires'] * np.random.uniform(0.05, 0.15, size=len(fact))).round(2)
    fact['effectif'] = np.random.randint(1, 25, size=len(fact))
    
    fact_finance = fact[['siren', 'id_geographie', 'id_secteur', 'date_creation', 'chiffre_affaires', 'resultat_net', 'effectif']]

    # --- CHARGEMENT ---
    dim_entreprise.to_sql('dim_entreprise', engine, if_exists='replace', index=False)
    dim_geo.to_sql('dim_geographie', engine, if_exists='replace', index=False)
    dim_secteur.to_sql('dim_secteur', engine, if_exists='replace', index=False)
    fact_finance.to_sql('fact_finance', engine, if_exists='replace', index=False)

    print("✅ Succès ! DWH complet et libellés reformulés.")

build_final_dwh_v2()

Extraction et nettoyage des données...
✅ Succès ! DWH complet et libellés reformulés.
